# LO benchmark (common-sense slices)

One notebook, **four published tasks**. Set `BENCHMARK_VERSION` in setup to one of:

| `BENCHMARK_VERSION` | Prompts | Task name |
|---|---|---|
| `needs common sense` | JSON implicit (9) | `needs_common_sense_accuracy` |
| `explicitly given common sense` | JSON explicit (9) | `explicitly_given_common_sense_accuracy` |
| `common sense problem audit` | needs-tacit A/B (9) | `common_sense_problem_audit_accuracy` |
| `common sense solution audit` | detects-violation A/B (9) | `common_sense_solution_audit_accuracy` |

**Kaggle:** Secrets → `GITHUB_TOKEN`. Internet ON. Run setup → dry-run → publish / Build for **that** version. Repeat with another `BENCHMARK_VERSION` to publish the other three.

**Outputs:** `lp_merged_results.csv` / `lp_rate_score_pivot.csv` under `/kaggle/working/`.

In [ ]:
import io
import shutil
import sys
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

GITHUB_OWNER = "FrancisGanong-N"
GITHUB_REPO = "sceptical_llms"
GITHUB_BRANCH = "master"
KAGGLE_REPO_DIR = Path("/kaggle/working") / GITHUB_REPO
FORCE_REPO_REFRESH = True
DEBUG_MAX_PROMPTS = None
MAX_OUTPUT_TOKENS = 1024
MODEL = "anthropic/claude-opus-5@default"
# MODEL = "openai/gpt-5.6-sol"

# One of the four LO slices (also used as the published task selector).
BENCHMARK_VERSION = "needs common sense"
# BENCHMARK_VERSION = "explicitly given common sense"
# BENCHMARK_VERSION = "common sense problem audit"
# BENCHMARK_VERSION = "common sense solution audit"


def download_repo_from_github() -> Path:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN").strip()
    url = (
        f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
        f"/zipball/{GITHUB_BRANCH}"
    )
    request = urllib.request.Request(
        url,
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "kaggle-sceptical-llms-lo-benchmark",
        },
    )

    staging = Path("/kaggle/working") / "_repo_staging"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir()

    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            with zipfile.ZipFile(io.BytesIO(response.read())) as archive:
                archive.extractall(staging)
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"GitHub download failed ({exc.code}) for "
            f"github.com/{GITHUB_OWNER}/{GITHUB_REPO}@{GITHUB_BRANCH}: {body[:300]}"
        ) from exc

    extracted = next(p for p in staging.iterdir() if p.is_dir())
    if KAGGLE_REPO_DIR.exists():
        shutil.rmtree(KAGGLE_REPO_DIR)
    shutil.copytree(extracted, KAGGLE_REPO_DIR)
    shutil.rmtree(staging)
    return KAGGLE_REPO_DIR


def bootstrap_repo() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "benchmarks" / "lp_rate_tasks.py").is_file():
            return candidate

    if Path("/kaggle/working").is_dir():
        tasks = KAGGLE_REPO_DIR / "benchmarks" / "lp_rate_tasks.py"
        if not FORCE_REPO_REFRESH and tasks.is_file():
            return KAGGLE_REPO_DIR
        return download_repo_from_github()

    raise RuntimeError(
        "Could not find sceptical-llms (need benchmarks/lp_rate_tasks.py). "
        "Run from the repo, or on Kaggle set GITHUB_TOKEN and Internet ON."
    )


ROOT = bootstrap_repo()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for module_name in list(sys.modules):
    if module_name == "benchmarks" or module_name.startswith("benchmarks."):
        del sys.modules[module_name]

from benchmarks.lp_rate import (
    LO_BENCHMARK_VERSIONS,
    example_ids_for_benchmark_version,
    task_name_for_version,
    task_slug_for_version,
)

if BENCHMARK_VERSION not in LO_BENCHMARK_VERSIONS:
    raise ValueError(
        f"BENCHMARK_VERSION={BENCHMARK_VERSION!r}; expected one of "
        f"{list(LO_BENCHMARK_VERSIONS)}"
    )

TASK_NAME = task_name_for_version(BENCHMARK_VERSION)
TASK_SLUG = task_slug_for_version(BENCHMARK_VERSION)
VERSION_PROMPT_IDS = example_ids_for_benchmark_version(
    benchmark_version=BENCHMARK_VERSION
)

print("Repo root:", ROOT)
print("BENCHMARK_VERSION:", BENCHMARK_VERSION)
print("TASK_NAME:", TASK_NAME)
print("TASK_SLUG:", TASK_SLUG)
print("Prompts in version:", len(VERSION_PROMPT_IDS))
print("MODEL:", MODEL)
print("DEBUG_MAX_PROMPTS:", DEBUG_MAX_PROMPTS)
print("MAX_OUTPUT_TOKENS:", MAX_OUTPUT_TOKENS)

In [ ]:
import pandas as pd
import kaggle_benchmarks as kbench
from IPython.display import FileLink, display

from benchmarks.lp_rate_tasks import evaluate_lp_rate_benchmark

(
    runs,
    score,
    merged_path,
    pivot_path,
    pivot,
    naive_rate,
    variant_scores,
) = evaluate_lp_rate_benchmark(
    kbench.llms[MODEL],
    max_prompts=DEBUG_MAX_PROMPTS,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    n_jobs=1,
    benchmark_version=BENCHMARK_VERSION,
)

print("BENCHMARK_VERSION:", BENCHMARK_VERSION)
print("Model:", MODEL)
print("Keyed accuracy:", f"{score.accuracy:.1%}")
print("Parse rate:", f"{score.parse_rate:.1%}")
print("Naive LP confusion rate:", f"{naive_rate:.1%}")
print("Merged results:", merged_path)

merged_df = pd.read_csv(merged_path)
resp_col = "llm_response" if "llm_response" in merged_df.columns else "response"
empty = int(merged_df[resp_col].astype(str).str.strip().eq("").sum())
print(f"Rows: {len(merged_df)}  |  empty {resp_col}: {empty}")

display(pivot)
display(
    FileLink(merged_path.name, result_html_prefix="Download merged: "),
    FileLink(pivot_path.name, result_html_prefix="Download pivot: "),
)
merged_df.head()

## Publish this version

Publishes **`TASK_NAME`** for the current `BENCHMARK_VERSION` (not `kbench.llm`).
To publish another slice: change `BENCHMARK_VERSION` in setup, re-run setup + this cell (and Build).

```powershell
python scripts/export_lo_kaggle_results.py --download --force-download
```

In [ ]:
import kaggle_benchmarks as kbench
import kaggle_benchmarks.ui.ipython_magics  # registers %choose

from benchmarks.lp_rate_tasks import resolve_lo_task

task = resolve_lo_task(BENCHMARK_VERSION)
print("Publishing:", TASK_NAME, "|", BENCHMARK_VERSION, "|", MODEL)
run = task.run(kbench.llms[MODEL])
print("Task score:", run.result)
print("Task passed:", run.passed)

# %choose needs the literal task name for this Build; keep in sync with TASK_NAME.
get_ipython().run_line_magic("choose", TASK_NAME)